In [2]:
import pandas as pd
import numpy as np

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [23]:
df=pd.read_csv("/Users/atharva/Downloads/LP3/Patil/ML/dataset/uber.csv")

In [7]:
print(df.head())

   Unnamed: 0                            key  fare_amount  \
0    24238194    2015-05-07 19:52:06.0000003          7.5   
1    27835199    2009-07-17 20:04:56.0000002          7.7   
2    44984355   2009-08-24 21:45:00.00000061         12.9   
3    25894730    2009-06-26 08:22:21.0000001          5.3   
4    17610152  2014-08-28 17:47:00.000000188         16.0   

           pickup_datetime  pickup_longitude  pickup_latitude  \
0  2015-05-07 19:52:06 UTC        -73.999817        40.738354   
1  2009-07-17 20:04:56 UTC        -73.994355        40.728225   
2  2009-08-24 21:45:00 UTC        -74.005043        40.740770   
3  2009-06-26 08:22:21 UTC        -73.976124        40.790844   
4  2014-08-28 17:47:00 UTC        -73.925023        40.744085   

   dropoff_longitude  dropoff_latitude  passenger_count  
0         -73.999512         40.723217                1  
1         -73.994710         40.750325                1  
2         -73.962565         40.772647                1  
3         

In [ ]:
#Remove missing values from the dataset
df=df.dropna()

In [9]:
#Remove Unnecessary Columns
if "key" in df.columns:
    df=df.drop("key",axis=1)

In [10]:
# 4. Remove invalid data
df = df[df["fare_amount"] > 0]
df = df[df["passenger_count"] > 0]

In [12]:
# 5. Remove outliers using IQR
Q1 = df["fare_amount"].quantile(0.25)
Q3 = df["fare_amount"].quantile(0.75)

IQR = Q3 - Q1

In [13]:
df = df[
    (df["fare_amount"] >= Q1 - 1.5 * IQR) &
    (df["fare_amount"] <= Q3 + 1.5 * IQR)
]

In [14]:
# 6. Convert pickup time
df["pickup_datetime"] = pd.to_datetime(
    df["pickup_datetime"]
)

In [15]:
df["hour"] = df["pickup_datetime"].dt.hour

In [16]:
# 7. Check correlation
print("\nCorrelation:")
print(df.select_dtypes(include="number").corr())


Correlation:
                   Unnamed: 0  fare_amount  pickup_longitude  pickup_latitude  \
Unnamed: 0           1.000000    -0.000100          0.000328        -0.001065   
fare_amount         -0.000100     1.000000          0.000125         0.002826   
pickup_longitude     0.000328     0.000125          1.000000        -0.811435   
pickup_latitude     -0.001065     0.002826         -0.811435         1.000000   
dropoff_longitude    0.000650    -0.002534          0.835878        -0.766797   
dropoff_latitude    -0.000286     0.000715         -0.850520         0.683972   
passenger_count      0.002175     0.011995         -0.000582        -0.001742   
hour                 0.001523    -0.013304          0.003668        -0.004779   

                   dropoff_longitude  dropoff_latitude  passenger_count  \
Unnamed: 0                  0.000650         -0.000286         0.002175   
fare_amount                -0.002534          0.000715         0.011995   
pickup_longitude            0.8

In [17]:
# 8. Select features and target
X = df[
    [
        "pickup_longitude",
        "pickup_latitude",
        "dropoff_longitude",
        "dropoff_latitude",
        "passenger_count",
        "hour"
    ]
]

y = df["fare_amount"]

In [18]:
# 9. Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [19]:
# 10. Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)

pred1 = lr.predict(X_test)

In [20]:
# 11. Random Forest
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

pred2 = rf.predict(X_test)

In [21]:
# 12. Evaluate models
print("\nLinear Regression")
print("R2:", r2_score(y_test, pred1))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred1)))

print("\nRandom Forest")
print("R2:", r2_score(y_test, pred2))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred2)))


Linear Regression
R2: 0.0004404112193429066
RMSE: 4.128596505319261

Random Forest
R2: 0.7200124335451303
RMSE: 2.1850806376143375


In [22]:
# 13. Predict Uber fare
print("\nPredicted Fare:", pred2[0])


Predicted Fare: 9.319000000000006
